# Toxic Comment Best Model Pipeline

This notebook trains the cleaned Logistic Regression pipeline, evaluates it, and saves the artifacts used by `src/main.py` for rapid comment classification.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split

if (Path.cwd() / 'src').exists():
    sys.path.insert(0, str((Path.cwd() / 'src').resolve()))
elif (Path.cwd() / 'toxic_pipeline.py').exists():
    sys.path.insert(0, str(Path.cwd().resolve()))

from toxic_pipeline import (
    BEST_LOGISTIC_REGRESSION_CONFIG,
    FEATURE_COLUMNS,
    LABEL_COLUMNS_TO_DROP,
    build_engineered_features,
    build_scaler,
    build_word_vectorizer,
    clean_text,
    predict_toxicity_raw_model,
    resolve_data_path,
    save_best_artifacts,
)

print('Imports OK')

Imports OK


In [2]:
SEARCH_RANDOM_STATE = 42
SEARCH_TEST_SIZE = 0.20
DATA_PATH = resolve_data_path()

print(f'Data path: {DATA_PATH}')
print(f'Feature columns ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}')
print(f'Best LR config: {BEST_LOGISTIC_REGRESSION_CONFIG}')

Data path: D:\Code\Repos\CPE232\project\data\train.csv
Feature columns (7): ['Question Mark Count', 'Profanity Count', 'Repeated Punctuation Count', 'Short/Unclear Without Toxic Signal Flag', 'Second-person Pronoun Count', 'URL Count', 'Non-toxic Negation Pattern Count']
Best LR config: {'penalty': 'l2', 'solver': 'liblinear', 'C': 1.0645493016479186, 'class_weight': {0: 1, 1: 3}, 'tol': 0.0002226958973431528, 'max_iter': 2000, 'random_state': 42, 'n_jobs': None}


In [3]:
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=LABEL_COLUMNS_TO_DROP)

df['raw_text'] = df['comment_text'].fillna('').astype(str)
df['clean_text'] = df['raw_text'].apply(clean_text)
df = df[df['clean_text'] != ''].copy()
df = df.drop_duplicates(subset=['clean_text']).reset_index(drop=True)

print(f'Dataset shape after cleaning: {df.shape}')
print(df['toxic'].value_counts())

Dataset shape after cleaning: (158194, 5)
toxic
0    143038
1     15156
Name: count, dtype: int64


In [4]:
feature_rows = [
    build_engineered_features(text).iloc[0].to_dict()
    for text in df['raw_text']
]
feature_df = pd.DataFrame(feature_rows, columns=FEATURE_COLUMNS)
df = pd.concat([df, feature_df], axis=1)

print('Engineered features ready')
print(df[FEATURE_COLUMNS].head(3))

Engineered features ready
   Question Mark Count  Profanity Count  Repeated Punctuation Count  \
0                    1                0                           0   
1                    0                0                           0   
2                    0                0                           0   

   Short/Unclear Without Toxic Signal Flag  Second-person Pronoun Count  \
0                                        0                            0   
1                                        0                            0   
2                                        0                            0   

   URL Count  Non-toxic Negation Pattern Count  
0          0                                 0  
1          0                                 0  
2          0                                 0  


In [5]:
X_raw = df['raw_text']
X_clean = df['clean_text']
X_eng = df[FEATURE_COLUMNS]
y = df['toxic']

(
    X_raw_train, X_raw_test,
    X_clean_train, X_clean_test,
    X_eng_train, X_eng_test,
    y_train, y_test,
) = train_test_split(
    X_raw, X_clean, X_eng, y,
    test_size=SEARCH_TEST_SIZE,
    random_state=SEARCH_RANDOM_STATE,
    stratify=y,
)

print(f'Train: {len(y_train):,} | Test: {len(y_test):,}')
print(f'Toxic ratio (train): {y_train.mean():.3f}')

Train: 126,555 | Test: 31,639
Toxic ratio (train): 0.096


In [6]:
word_vec = build_word_vectorizer()
scaler = build_scaler()

X_word_train = word_vec.fit_transform(X_clean_train)
X_word_test = word_vec.transform(X_clean_test)
X_eng_train_scaled = csr_matrix(scaler.fit_transform(X_eng_train.values))
X_eng_test_scaled = csr_matrix(scaler.transform(X_eng_test.values))

X_train = hstack([X_word_train, X_eng_train_scaled], format='csr')
X_test = hstack([X_word_test, X_eng_test_scaled], format='csr')

print(f'Word TF-IDF shape: {X_word_train.shape}')
print(f'Final train matrix: {X_train.shape}')
print(f'Final test matrix : {X_test.shape}')

Word TF-IDF shape: (126555, 10000)
Final train matrix: (126555, 10007)
Final test matrix : (31639, 10007)


In [7]:
best_model = LogisticRegression(**BEST_LOGISTIC_REGRESSION_CONFIG)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print('=== Logistic Regression Report ===')
print(classification_report(y_test, y_pred, target_names=['Not Toxic', 'Toxic']))
print(f'ROC-AUC : {roc_auc_score(y_test, y_prob):.4f}')
print(f'F1 (Toxic): {f1_score(y_test, y_pred):.4f}')

d:\Code\Repos\CPE232\project\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


=== Logistic Regression Report ===
              precision    recall  f1-score   support

   Not Toxic       0.97      0.98      0.98     28608
       Toxic       0.78      0.75      0.76      3031

    accuracy                           0.96     31639
   macro avg       0.88      0.86      0.87     31639
weighted avg       0.95      0.96      0.96     31639

ROC-AUC : 0.9636
F1 (Toxic): 0.7631


In [8]:
artifact_paths = save_best_artifacts(
    model=best_model,
    word_vectorizer=word_vec,
    scaler=scaler,
    metadata={
        'test_f1': float(f1_score(y_test, y_pred)),
        'test_roc_auc': float(roc_auc_score(y_test, y_prob)),
    },
)

for name, path in artifact_paths.items():
    print(f'Saved {name}: {path}')

Saved model: D:\Code\Repos\CPE232\project\src\best_model_final.pkl
Saved word_vectorizer: D:\Code\Repos\CPE232\project\src\best_model_word_vectorizer.pkl
Saved scaler: D:\Code\Repos\CPE232\project\src\best_model_scaler.pkl
Saved metadata: D:\Code\Repos\CPE232\project\src\best_model_metadata.pkl


In [9]:
artifacts = {
    'model': best_model,
    'word_vectorizer': word_vec,
    'scaler': scaler,
}

samples = [
    'You are such an idiot, go fuck yourself!',
    'I do not think this policy is fair to everyone.',
    'Love you',
    "I do not like it when people are rude to me.",
    "I don't like your car.",
    "I do not hate you.",
    "you"
]

for sample in samples:
    result = predict_toxicity_raw_model(sample, artifacts)
    print(f"{result['label']:10s} ({result['probability']:.2%})  ->  {sample}")

Toxic      (100.00%)  ->  You are such an idiot, go fuck yourself!
Not Toxic  (1.38%)  ->  I do not think this policy is fair to everyone.
Not Toxic  (1.84%)  ->  Love you
Toxic      (62.85%)  ->  I do not like it when people are rude to me.
Not Toxic  (12.08%)  ->  I don't like your car.
Not Toxic  (14.89%)  ->  I do not hate you.
Not Toxic  (3.34%)  ->  you
